# Asistente RAG sobre documentación técnica minera — demo del Ejercicio C-2

Este notebook ejecuta el flujo completo del asistente sobre los tres PDF del enunciado y se
entrega **con sus salidas**. No contiene lógica propia: cada celda llama a una clase de
`rag_minero` que tiene sus pruebas, en el orden que menos cuesta: primero todo lo que es
gratis y determinista (leer, trocear, calibrar la puerta de dominio, la ablación de chunking)
y al final lo que llama a un modelo (respuestas, preguntas de control, RAGAS).

Reproducir: activar `.venv-rag`, exportar `RAG_PDF_DIR` con el directorio de los PDF, el perfil
de la CLI de Databricks y las variables `RAG_*` que describe el README, y ejecutar
`jupyter nbconvert --to notebook --execute --inplace rag_demo.ipynb`.

Sobre los modelos: el workspace de prueba apaga los modelos propietarios con un límite de tasa
cero, así que esta corrida usa **Qwen3-Next 80B** como generador y **Llama 3.3 70B** como juez
de RAGAS, dos familias distintas. Volver a Claude Sonnet 5 es cambiar `RAG_MODELO_GENERADOR`
y `RAG_MODELO_JUEZ`; ninguna línea de código.

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from rag_minero.documentos import TipoElemento
from rag_minero.flujo import Configuracion, Flujo

configuracion = Configuracion.desde_entorno()
flujo = Flujo(configuracion)
print("almacen:", configuracion.almacen)
print("generador:", configuracion.modelo_generador)
print("juez RAGAS:", configuracion.modelo_juez)
print("embeddings:", configuracion.modelo_embeddings)
print("tope de tokens de la sesion:", f"{configuracion.tokens_maximos:,}")

## 1. Lectura de los PDF: elementos tipados y tablas partidas

El lector no devuelve texto plano sino elementos con tipo —encabezado, prosa, paso, tabla,
advertencia— porque el chunking decide por tipo de elemento. Las tablas que cruzan páginas
llegan fusionadas: la de verificaciones pre-turno tiene la cabecera en la página 1 y las
filas en la 2, y aquí aparece como una sola tabla de seis filas.

In [ ]:
documentos = flujo.cargar_documentos()
for d in documentos:
    print(f"{d.codigo:<28} {d.tipo.value:<14} v{d.version:<8} {d.fecha}  {d.clasificacion:<12} {d.titulo[:60]}")
    conteo = {t.value: len(d.de_tipo(t)) for t in TipoElemento if d.de_tipo(t)}
    print(f"    elementos: {conteo}")
    print(f"    tablas (filas): {[len(t.filas) for t in d.tablas]}")

## 2. Chunking por género y por elemento, con las dos variantes de control

Tres estrategias, una por género, más dos de control para la ablación: solo sección y tamaño
fijo. Cada chunk lleva delante el documento y la sección, y en metadatos los códigos y frentes
que menciona, porque en mina se pregunta por código.

In [ ]:
chunks = flujo.trocear()
print("chunks por variante:", flujo.resultados.chunks_por_variante)
muestras = [c for c in chunks if c.metadata.get("fila") in ("CP-03", "AC-L8-BP-2241") or c.metadata.get("prioridad") == "alta"]
for c in muestras:
    print(f"\n--- {c.id}  codigos={c.metadata['codigos']!r}")
    print(c.page_content[:320])
tabla = next(c for c in chunks if c.metadata.get("filas") == 5)
print(f"\n--- {tabla.id}  frentes={tabla.metadata['frentes']!r}  (tabla entera, {len(tabla.page_content)} caracteres)")
print(tabla.page_content[:260])

## 3. Puerta de dominio: umbral calibrado, no fijado a mano

La cobertura léxica de una pregunta es la fracción de sus términos de contenido que existe en el
corpus. El umbral se deriva del golden set y de las diez preguntas fuera de dominio.

In [ ]:
calibracion = flujo.calibrar()
print(f"umbral de cobertura: {calibracion.cobertura_minima:.3f}")
print(f"golden set aceptado: {calibracion.aceptadas_del_dominio}/{calibracion.total_del_dominio}")
print(f"fuera de dominio rechazado: {calibracion.rechazadas_fuera}/{calibracion.total_fuera}")
assert flujo.vocabulario is not None
for caso in flujo.golden.casos:
    print(f"  {caso.id:<11} {flujo.vocabulario.cobertura(caso.pregunta):.2f}  {caso.pregunta[:70]}")
for pregunta in flujo.control.fuera_de_dominio[:4]:
    print(f"  {'fuera':<11} {flujo.vocabulario.cobertura(pregunta):.2f}  {pregunta[:70]}")

## 4. Ablación de chunking, sin modelo juez

Para cada pregunta del golden set se mide si los pasajes de referencia aparecen entre los seis
chunks recuperados: precisión de contexto con la fórmula sin LLM de RAGAS y recall de referencias.
Los embeddings son los reales (`qwen3-embedding-0-6b`), el almacén es Chroma con BM25, y no se
gasta un token de generación. Si la estrategia por género y elemento no superara a las de
control, este notebook lo mostraría.

In [ ]:
t0 = time.time()
ablacion = flujo.ablacion()
print(flujo.resultados.ablacion_tabla)
print(f"\n{len(ablacion)} mediciones en {time.time() - t0:.0f} s")

## 5. Índice, asistente y evaluación

A partir de aquí se gasta. El almacén configurado se construye —en Databricks, el endpoint de
Vector Search se crea si no existe y se borra en el cierre—, el asistente responde las trece
preguntas de control y las diez del golden set, y RAGAS juzga cada respuesta. Todo dentro de
un `try/finally` para que el cierre corra aunque algo falle a mitad de camino.

In [ ]:
t0 = time.time()
almacen = flujo.construir_almacen()
print(f"almacen {almacen.nombre}: {almacen.cantidad} chunks indexados en {time.time() - t0:.0f} s")
asistente = flujo.construir_asistente(almacen)
for pregunta in ("H-HIDRA-05", "¿qué frente tuvo la mayor ley media?"):
    print(f"\nrecuperacion para {pregunta!r}:")
    for r in asistente.recuperar(pregunta)[:3]:
        print(f"   {r.score:.3f} {r.origen:<8} {r.chunk_id}")

In [ ]:
try:
    control = flujo.probar_control(asistente)
    print("Preguntas de control")
    for r in control:
        estado = "RECHAZADA" if r.rechazada else ("BLOQUEADA" if r.bloqueada else "respondida")
        print(f"  [{estado:<10}] {r.pregunta[:58]:<58} | {r.motivo[:48]}")
    print("\nRespuestas a las preguntas del dominio sin respaldo documental:")
    for r in control[-3:]:
        print("  -", r.texto.replace(chr(10), " ")[:220])

    t0 = time.time()
    ragas = flujo.evaluar_ragas(asistente)
    print(f"\nRAGAS en {time.time() - t0:.0f} s")
    print(flujo.resultados.ragas_tabla)
    print({k: round(v, 3) for k, v in flujo.resultados.ragas_resumen.items()})
finally:
    ruta = flujo.cerrar(almacen)
    print(f"\nresultados en {ruta}")
    print(f"tokens consumidos: {flujo.presupuesto.consumidos:,} en {flujo.presupuesto.llamadas} llamadas")

## 6. Las respuestas, con sus citas

Cada respuesta cita el identificador del chunk que la sostiene. El verificador de hechos ya
contrastó cada cifra y cada código contra esos pasajes: lo que se lee aquí pasó ese filtro.

In [ ]:
for r in ragas:
    print(f"\n[{r.caso_id}] {r.respuesta.pregunta}")
    print("   ", r.respuesta.texto.replace(chr(10), " "))
    print("    fuentes:", ", ".join(r.respuesta.fuentes[:3]), "...")